# Evaluation Metrics — Classification (6/19/26)

---

*Codecademy — MLE Path: ML Fundamentals. Concise grad-student notes.*

> **Scope of this lesson:** how to judge a *classifier* once it's trained. The single number "accuracy" is often a trap; this lesson builds the **confusion matrix** and the four metrics derived from it (accuracy, recall, precision, F1).

## Confusion Matrix — the 4 outcomes

> **Prerequisite:** measuring predictive power happens *after* splitting into **train / validation / test** — you compute these on the held-out evaluation set, never on training data. (See `Train_Validation_Test_Lesson.ipynb`.)

Pass the evaluation set's features through the trained model → list of **predictions**. Compare each to the **actual label**. Every comparison lands in one of **four buckets** (spam-classifier framing):

- **True Positive (TP):** predicted spam, *was* spam ✅
- **True Negative (TN):** predicted not-spam, *was* not-spam ✅
- **False Positive (FP):** predicted spam, was *not* spam ❌ (false alarm)
- **False Negative (FN):** predicted not-spam, *was* spam ❌ (miss)

Read each as *"<True/False> <what the model predicted>"*: **False Positive** = model said positive (spam) and was wrong.

### The confusion matrix (Codecademy layout)
Predicted classes = **columns**, actual classes = **rows**:

| | Predicted − | Predicted + |
|---|---|---|
| **Actual −** | TN | FP |
| **Actual +** | FN | TP |

Everything below (accuracy, recall, precision, F1) is just arithmetic on these four counts.

## Accuracy — the naive default

Correctly classified predictions (TP **and** TN) over *all* predictions:

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} = \frac{\text{correct}}{\text{total}}$$

- For our exercise data: $\frac{TP+TN}{\text{total}} = \frac{3+0}{10} = 0.30$ — a weak classifier.
- Intuitive, but **misleading on imbalanced data**. *Class-imbalance trap:* if 99% of emails are not-spam, a model that always predicts "not spam" scores **99% accuracy** while catching *zero* spam. Looks great, useless.
- Accuracy weights every error equally and ignores *which* class you fail on → motivates recall & precision next.

In [ ]:
# Accuracy exercise -- your solution (helper fn), reusing the counters from above
def Acc(TP, TN, FP, FN):
    Acc = (TP + TN) / len(predicted)   # len(predicted) == total predictions
    return Acc

accuracy = Acc(true_positives, true_negatives, false_positives, false_negatives)
print(accuracy)   # 0.3

## Recall — "of the real positives, how many did we catch?"

$$\text{Recall} = \frac{TP}{TP + FN}$$

- Denominator = **all actual positives**. Answers: *how complete is the catch?*
- Also called **sensitivity** / **true positive rate**.
- **High recall matters when missing a positive is costly** — disease screening, fraud detection (a missed cancer is far worse than a false alarm).
- Gameable: predict *everything* positive → recall = 1.0 (but precision tanks). So never report it alone.

## Precision — "of our positive calls, how many were right?"

$$\text{Precision} = \frac{TP}{TP + FP}$$

- Denominator = **all predicted positives**. Answers: *how trustworthy is a positive prediction?*
- **High precision matters when a false alarm is costly** — spam filter (don't trash a real email), recommending content.
- Precision vs recall is a **trade-off**: tightening the threshold to be "sure" raises precision but lowers recall, and vice versa.

## F1 Score — one number that balances the two

$$F_1 = 2 \cdot \frac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$$

- The **harmonic mean** of precision and recall (not the arithmetic mean — that's the key trick).
- **Why harmonic?** It punishes imbalance. If precision = 1.0 but recall = 0.0, arithmetic mean = 0.5 (looks ok), but **F1 = 0**. F1 is only high when *both* are high.
- Ranges 0→1. Use it as a single summary when you care about precision and recall *together* and classes are imbalanced.

## Exercise — count the four outcomes & build the matrix

*Codecademy exercise.* `actual` = true labels (1 = spam, 0 = not spam); `predicted` = the classifier's calls.
Steps: (1) init four counters to 0 → (2) loop, `+1` to the matching bucket per email → (3) print them → (4) build the confusion matrix with sklearn.

In [ ]:
from sklearn.metrics import confusion_matrix

actual    = [1, 0, 0, 1, 1, 1, 0, 1, 1, 1]
predicted = [0, 1, 1, 1, 1, 0, 1, 0, 1, 0]

# initializing confusion-matrix elements
true_positives  = 0
true_negatives  = 0
false_positives = 0
false_negatives = 0

for i in range(len(predicted)):
    if actual[i] == 1 and predicted[i] == 1:   # predicted spam, was spam
        true_positives += 1
    if actual[i] == 0 and predicted[i] == 0:   # predicted not-spam, was not-spam
        true_negatives += 1
    if actual[i] == 0 and predicted[i] == 1:   # predicted spam, was not-spam (false alarm)
        false_positives += 1
    if actual[i] == 1 and predicted[i] == 0:   # predicted not-spam, was spam (miss)
        false_negatives += 1

print(true_positives)    # 3
print(true_negatives)    # 0
print(false_positives)   # 3
print(false_negatives)   # 4

# scikit-learn builds it for you -> layout [[TN, FP], [FN, TP]]
conf_matrix = confusion_matrix(actual, predicted)
print(conf_matrix)       # [[0 3]
                         #  [4 3]]

## The scikit-learn way (what you'll actually use)

In [ ]:
from sklearn.metrics import (accuracy_score, recall_score,
                             precision_score, f1_score,
                             classification_report)

# reuse `actual` / `predicted` from the exercise above
print("accuracy :", round(accuracy_score(actual, predicted), 3))
print("recall   :", round(recall_score(actual, predicted), 3))
print("precision:", round(precision_score(actual, predicted), 3))
print("f1       :", round(f1_score(actual, predicted), 3))

print("\n", classification_report(actual, predicted))  # all metrics per class at once

> **sklearn `confusion_matrix` layout gotcha:** rows = actual, cols = predicted, ordered `[0, 1]`. So it's `[[TN, FP], [FN, TP]]` — *not* the textbook layout above. Always sanity-check which corner is which.

---
## Q & A (captured as I go)

*Questions I posed during the lesson + answers, folded in per sublesson.*

_(none yet — will fill in as we work through the lesson)_

---
## TL;DR

- Everything comes from the **confusion matrix**: TP / FP / FN / TN.
- **Accuracy** = overall correctness; lies on imbalanced data.
- **Recall** = `TP/(TP+FN)` — caught positives; maximize when *misses* are costly.
- **Precision** = `TP/(TP+FP)` — trustworthy positives; maximize when *false alarms* are costly.
- **F1** = harmonic mean of the two; high only when *both* are high. Use for imbalanced classes.
- In practice: `from sklearn.metrics import ...` + `classification_report` for all of it at once.